# Credit Risk — End-to-End Pipeline Notebook

**Purpose:** Exploratory analysis, leakage-safe preprocessing, panel feature engineering, grouped cross-validation, and calibrated modeling for the `gb` default target.

**Data:** `train_df.csv` — 26,824 rows x 554 columns, 5,243 entities, ~2.21% positive class.

---

## Notebook map

1. Introduction & Configuration
2. EDA & Data Cleaning (visual, explicit)
3. Feature Engineering
4. Validation Strategy
5. Model Training (CatBoost & LightGBM)
6. Post-Processing, Calibration & F1 Threshold Tuning
7. Key Insights, Metrics Summary & Future Roadmap


## Section 1: Introduction & Configuration


In [ ]:
# Environment check — requires Python 3.11+ with catboost & lightgbm
import sys
for pkg in ("catboost", "lightgbm", "sklearn", "seaborn"):
    __import__(pkg)
print(f"Python {sys.version.split()[0]} | dependencies OK")


## Section 3: Feature Engineering

Panel-aware features fitted inside each CV fold on training rows only:

- MNAR indicators: `{col}__is_missing`
- ID aggregations: mean / std / min / max per entity
- Temporal deltas: consecutive-row diff within each id


In [ ]:
def sanitize_feature_names(columns):
    return [re.sub(r"[^\w]", "_", c) for c in columns]


def run_fold_pipeline(df, train_idx, val_idx):
    train_df = df.iloc[train_idx].copy()
    val_df = df.iloc[val_idx].copy()

    cleaner = DataCleaner(config)
    cleaner.fit(train_df, train_df[TARGET_COL])
    train_clean = cleaner.transform(train_df)
    val_clean = cleaner.transform(val_df)

    base_features = cleaner.get_feature_columns()
    cat_feature_cols = [c for c in base_features if c.startswith(config.cat_feature_prefix)]

    panel = PanelFeatureEngineer(config)
    panel.fit(train_clean, base_features)
    train_eng = panel.transform(train_clean)
    val_eng = panel.transform(val_clean)

    all_features = [c for c in base_features + panel.engineered_column_names() if c in train_eng.columns]

    encoder = CategoricalEncoder(cat_feature_cols)
    encoder.fit(train_eng)
    train_enc = encoder.transform(train_eng)
    val_enc = encoder.transform(val_eng)

    cat_idx = encoder.cat_feature_indices(all_features)
    return train_enc, val_enc, all_features, cat_idx


def to_xy(frame, features):
    X = frame[features].copy()
    X.columns = sanitize_feature_names(features)
    return X, frame[TARGET_COL]

print("Fold pipeline helpers defined.")


## Section 4: Validation Strategy

StratifiedGroupKFold (5 folds) on stable entity IDs. Volatile IDs are excluded from validation but kept in training.


In [ ]:
cv_splits = build_group_cv_splits(
    df, ID_COL, TARGET_COL, volatile_ids_list,
    n_splits=config.n_splits,
    random_state=config.random_state,
    exclude_volatile_from_val=config.exclude_volatile_ids_from_validation,
)

split_summary = pd.DataFrame([
    {"fold": s.fold, "train_rows": len(s.train_idx), "val_rows": len(s.val_idx),
     "volatile_in_train": s.n_volatile_in_train, "val_groups": s.n_stable_val_groups}
    for s in cv_splits
])
display(split_summary)


## Section 5: Model Training (CatBoost & LightGBM)

Grouped CV with per-fold preprocessing. Both models handle missing values and class imbalance natively.


In [ ]:
oof_preds = {m: np.full(len(df), np.nan) for m in MODELS}
fold_metrics = []

for split in cv_splits:
    train_enc, val_enc, features, cat_idx = run_fold_pipeline(df, split.train_idx, split.val_idx)
    X_train, y_train = to_xy(train_enc, features)
    X_val, y_val = to_xy(val_enc, features)

    for model_name in MODELS:
        trainer = build_model_trainer(model_name, config)
        trainer.fit(X_train, y_train, X_val, y_val, cat_features=cat_idx)
        val_prob = trainer.predict_proba(X_val)
        oof_preds[model_name][split.val_idx] = val_prob

        m = compute_metrics(y_val.values, val_prob, config.precision_at_k_fraction)
        fold_metrics.append({"fold": split.fold, "model": model_name, **m})
        print(
            f"Fold {split.fold} | {model_name:9s} | PR-AUC={m['pr_auc']:.4f} "
            f"ROC-AUC={m['roc_auc']:.4f} P@10%={m['precision_at_k']:.4f} Brier={m['brier']:.4f}"
        )

fold_metrics_df = pd.DataFrame(fold_metrics)
display(fold_metrics_df)


In [ ]:
cv_summary = (
    fold_metrics_df.groupby("model")
    .agg(
        pr_auc_mean=("pr_auc", "mean"), pr_auc_std=("pr_auc", "std"),
        roc_auc_mean=("roc_auc", "mean"), roc_auc_std=("roc_auc", "std"),
        precision_at_k_mean=("precision_at_k", "mean"),
        brier_mean=("brier", "mean"),
    )
    .round(4)
)
print("Cross-validation summary (raw fold metrics):")
display(cv_summary)


## Section 6: Post-Processing, Probability Calibration & F1 Threshold Tuning

1. Isotonic calibration on out-of-fold predictions
2. F1-maximizing threshold search on calibrated OOF scores
3. Evaluation plots


In [ ]:
def find_optimal_f1_threshold(y_true, y_prob, n_thresholds=200):
    thresholds = np.linspace(0.001, 0.999, n_thresholds)
    best_f1, best_t = 0.0, 0.5
    for t in thresholds:
        f1 = f1_score(y_true, (y_prob >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1


y_all = df[TARGET_COL].values
calibrators = {}
oof_calibrated = {}
threshold_results = []

for model_name in MODELS:
    oof_raw = oof_preds[model_name]
    valid = ~np.isnan(oof_raw)

    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(oof_raw[valid], y_all[valid])
    calibrators[model_name] = iso

    oof_cal = iso.predict(oof_raw[valid])
    oof_calibrated[model_name] = np.full(len(df), np.nan)
    oof_calibrated[model_name][valid] = oof_cal

    opt_t, max_f1 = find_optimal_f1_threshold(y_all[valid], oof_cal)

    cal_m = compute_metrics(y_all[valid], oof_cal, config.precision_at_k_fraction)

    threshold_results.append({
        "model": model_name,
        "optimal_threshold": opt_t,
        "max_f1_score": max_f1,
        **{f"oof_{k}": v for k, v in cal_m.items()},
    })

threshold_df = pd.DataFrame(threshold_results)
print("OOF metrics with calibration and F1-optimal thresholds:")
display(threshold_df.round(4))


In [ ]:
final_results = []
for model_name in MODELS:
    row = threshold_df[threshold_df["model"] == model_name].iloc[0]
    final_results.append({
        "Model": model_name,
        "PR-AUC (OOF cal)": row["oof_pr_auc"],
        "ROC-AUC (OOF cal)": row["oof_roc_auc"],
        "Precision@10%": row["oof_precision_at_k"],
        "Brier (OOF cal)": row["oof_brier"],
        "Optimal Threshold": row["optimal_threshold"],
        "Max F1-Score": row["max_f1_score"],
    })

final_results_df = pd.DataFrame(final_results).set_index("Model")
print("=== FINAL CV RESULTS TABLE ===")
display(final_results_df.round(4))
final_results_df.round(4).to_csv(OUTPUT_DIR / "final_cv_results.csv")
fold_metrics_df.to_csv(OUTPUT_DIR / "cv_fold_metrics.csv")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, model_name in zip(axes, MODELS):
    oof_raw = oof_preds[model_name]
    valid = ~np.isnan(oof_raw)
    oof_cal = oof_calibrated[model_name][valid]

    for probs, label in [(oof_raw[valid], "Raw"), (oof_cal, "Calibrated")]:
        frac_pos, mean_pred = calibration_curve(y_all[valid], probs, n_bins=10, strategy="quantile")
        ax.plot(mean_pred, frac_pos, marker="o", label=label)
    ax.plot([0, 1], [0, 1], "--", color="gray")
    ax.set_title(f"{model_name} — Reliability Diagram")
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Fraction of positives")
    ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "calibration_curves.png", dpi=120)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for model_name, color in zip(MODELS, [PALETTE["primary"], PALETTE["secondary"]]):
    valid = ~np.isnan(oof_calibrated[model_name])
    prec, rec, _ = precision_recall_curve(y_all[valid], oof_calibrated[model_name][valid])
    ap = average_precision_score(y_all[valid], oof_calibrated[model_name][valid])
    ax.plot(rec, prec, label=f"{model_name} (AP={ap:.4f})", color=color)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("OOF Precision-Recall Curves (Calibrated)")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "oof_pr_curves.png", dpi=120)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
thresholds = np.linspace(0.001, 0.999, 150)
for model_name, color in zip(MODELS, [PALETTE["primary"], PALETTE["secondary"]]):
    valid = ~np.isnan(oof_calibrated[model_name])
    y_v = y_all[valid]
    p_v = oof_calibrated[model_name][valid]
    f1s = [f1_score(y_v, (p_v >= t).astype(int), zero_division=0) for t in thresholds]
    ax.plot(thresholds, f1s, label=model_name, color=color)
    opt_t = threshold_df.loc[threshold_df["model"] == model_name, "optimal_threshold"].iloc[0]
    ax.axvline(opt_t, color=color, ls="--", alpha=0.7)
ax.set_xlabel("Probability threshold")
ax.set_ylabel("F1-score")
ax.set_title("F1-Score vs Threshold (Calibrated OOF)")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "f1_threshold_curves.png", dpi=120)
plt.show()


In [ ]:
volatile_set = set(volatile_ids_list)
stable = df[~df[ID_COL].isin(volatile_set)]
entity_y = stable.groupby(ID_COL)[TARGET_COL].first()
val_entities = np.random.RandomState(RANDOM_STATE).choice(
    entity_y.index.values, size=max(1, int(0.15 * len(entity_y))), replace=False,
)
val_mask = df[ID_COL].isin(val_entities)
train_idx = df.index[~val_mask].to_numpy()
val_idx = df.index[val_mask].to_numpy()

train_enc, val_enc, features, cat_idx = run_fold_pipeline(df, train_idx, val_idx)
X_train, y_train = to_xy(train_enc, features)
X_val, y_val = to_xy(val_enc, features)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, model_name in zip(axes, MODELS):
    trainer = build_model_trainer(model_name, config)
    trainer.fit(X_train, y_train, X_val, y_val, cat_features=cat_idx)
    imp = trainer.feature_importance(features).sort_values(ascending=True).tail(25)
    imp.plot(kind="barh", ax=ax, color=PALETTE["primary"] if model_name == "catboost" else PALETTE["secondary"])
    ax.set_title(f"{model_name} — Top 25 Feature Importance (Gain)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "feature_importance.png", dpi=120)
plt.show()


## Section 7: Key Insights, Metrics Summary & Future Roadmap

### Key insights

| Finding | Modeling implication |
|---------|-------------------|
| 2.21% positive rate | Optimize PR-AUC and F1 with tuned threshold; use class weights |
| Panel data (~5 rows/ID) | StratifiedGroupKFold mandatory |
| 47.5% static features | Identity leakage risk — group isolation enforced |
| 88 MNAR signals | Missing indicators added per fold |
| 702+ correlated pairs | MI/variance pruning per fold |
| 13 volatile IDs | Excluded from validation folds |

### Metrics tracked

- PR-AUC (primary)
- ROC-AUC
- Precision@10%
- Brier score (calibration)
- F1-score with OOF threshold tuning on calibrated probabilities

### Future roadmap

1. Lag and rolling features on dynamic monotonic numerics.
2. Entity-level model with one row per ID.
3. SHAP monitoring for production drift.
4. Cost-sensitive threshold once FP/FN costs are defined.
5. Locked hold-out entity set for deployment sign-off.
